[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/conformal_elliptical/conformal_elliptical.ipynb)

# The Planes Across the Cube's Rotation Axes

A cube turns into itself about thirteen axes: three through the centres of opposite faces, six through the midpoints of opposite edges, and four through opposite corners. The plane through the centre across each axis cuts the sphere around the cube in a great circle, and every rotation of the cube carries these thirteen great circles onto one another. Nine of the planes, those across the face and edge axes, are also the cube's mirror planes.

Each plane is a vector, and a point of the sphere an antivector; their pairing, `plane & point`, is the point's height above the plane, zero on its great circle. A rotor carries a plane by its sandwich, `rotor >> planes`, and the metric pairing of two unit planes, `a | b`, is plus or minus one exactly when they coincide, up to sense.

In the notation of crystallography the thirteen axes are the cube's four-fold, two-fold and three-fold axes, of the rotation group $O$, and the nine mirror planes those of the full point group $O_h$ in the notation of Schoenflies.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np
from numga import NumpyContext

from examples.util.animation import save_figure
from examples.quadrics.conformal_elliptical import render
# The algebra, shared with the example's renderer.
from examples.quadrics.conformal_elliptical.core import ga

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()
# One direction along each rotation axis of the cube: through its faces, its edges and its corners.
FACE_AXES = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
EDGE_AXES = [[1, 1, 0], [1, -1, 0], [1, 0, 1], [1, 0, -1], [0, 1, 1], [0, 1, -1]]
CORNER_AXES = [[1, 1, 1], [1, 1, -1], [1, -1, 1], [-1, 1, 1]]

## 1. The planes

One direction along each axis gives the plane across it, normalized. Each plane's great circle is drawn on the sphere.

In [ ]:
def planes_across_axes() -> Vector:
    """The unit plane across each rotation axis of the cube, through its centre."""
    return mv.vector(FACE_AXES + EDGE_AXES + CORNER_AXES).normalized()                 # [planes] Vector



planes = planes_across_axes()                                                   # [planes] Vector
save_figure(render.draw_planes(planes), "cube_axis_planes");

## 2. Carried onto one another

A quarter turn about a face axis and a third of a turn about a corner axis generate all the cube's rotations. Each carries every plane onto one of the thirteen: the largest of a turned plane's pairings with the planes is one.

In [ ]:
quarter = (mv.xy * (np.pi / 4)).exp()                                           # [] Rotor
third = ((mv.xy + mv.yz + mv.zx).normalized() * (np.pi / 3)).exp()             # [] Rotor
alignments = [((rotor >> planes)[:, None] | planes).abs() for rotor in (quarter, third)]   # [planes, planes] Scalar each

In [ ]:
for name, alignment in zip(("quarter turn", "third of a turn"), alignments):
    print(name, "largest pairing of each turned plane:", alignment.to_array().max(axis=1))

In [ ]:
# checks
# Each generating rotation carries every plane onto one of the planes, up to its sense.
for alignment in alignments:
    np.testing.assert_allclose(alignment.to_array().max(axis=1), 1.0, atol=1e-12)